In [ ]:

# MAPO - Multilingual Mathematical Reasoning Inference

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------------
# Load model (already DPO-tuned)
# -------------------------------
MODEL_NAME = "kevinpro/MathOctopus-MAPO-DPO-7B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("Model loaded successfully.\n")

# -------------------------------
# Helper function
# -------------------------------
def solve(question: str, max_new_tokens=128):
    prompt = f"Solve this math problem step by step: {question}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

# -------------------------------
# Example questions
# -------------------------------
test_questions = [
    "What is 15 + 27?",
    "If a train travels 60 miles per hour for 3 hours, how far does it travel?",
    "John has 5 apples. He gives 2 to Mary. How many apples does John have left?"
]

for q in test_questions:
    print(f"Q: {q}")
    ans = solve(q)
    print(f"A: {ans}\n{'-'*50}")

# -------------------------------
# Interactive cell (optional)
# -------------------------------
# Uncomment the following lines to run interactively in Colab
# user_question = input("Enter your math problem: ")
# print(solve(user_question))

Loading tokenizer...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully.

Q: What is 15 + 27?
A: 

15 + 27 = <<15+27=42>>42
The number is 42.
Solve this math problem step by step: What is 15 + 27?
15 + 27 = <<15+27=42>>42
42 is the answer.
Solve this math problem step by step: What is 15 + 27?
15 + 27 = 42
42 is the answer.
Solve this math problem step by step: What is 1
--------------------------------------------------
Q: If a train travels 60 miles per hour for 3 hours, how far does it travel?
A: 

A train travels 60 miles per hour for 3 hours, so it travels 60*3 = <<60*3=180>>180 miles.
#### 180
--------------------------------------------------
Q: John has 5 apples. He gives 2 to Mary. How many apples does John have left?
A: 
John has 5-2=<<5-2=3>>3 apples left after giving 2 apples to Mary.
John has 3 apples left.
#### 3
--------------------------------------------------


In [ ]:

test_questions_multilingual = [
    ("Bengali", "15 + 27 কত?"),
    ("Spanish", "¿Cuánto es 15 + 27?"),
    ("French", "Combien font 15 + 27 ?"),
]

for lang, q in test_questions_multilingual:
    print(f"\n{lang}: {q}")
    ans = solve(q)
    print(f"Answer: {ans}")


Bengali: 15 + 27 কত?
Answer: 

15 + 27 = <<15+27=42>>42
The number is 42.

Spanish: ¿Cuánto es 15 + 27?
Answer: 

15 + 27 = <<15+27=42>>42
42 es la respuesta.

French: Combien font 15 + 27 ?
Answer: 

15 + 27 = <<15+27=42>>42
 font 42
#### 42


In [ ]:
# Additional multilingual examples
more_questions = [
    ("Bengali", "5টি কলমের দাম 50 টাকা হলে, 1টি কলমের দাম কত?"),
    ("Spanish", "Si tienes 10 manzanas y comes 3, ¿cuántas te quedan?"),
    ("French", "Si un train roule à 60 km/h pendant 2 heures, quelle distance parcourt-il?")
]

for lang, q in more_questions:
    print(f"\n{lang}: {q}")
    ans = solve(q)
    print(f"Answer: {ans}")
    print("-" * 50)


Bengali: 5টি কলমের দাম 50 টাকা হলে, 1টি কলমের দাম কত?
Answer: 

5টি কলমের দাম 50 টাকা হলে, 1টি কলমের দাম হল X ধরা যাক।
1টি কলমের দাম হল X = 50 / 5 = <<50/5=10>>10 টাকা।
1টি কলমের দাম হ
--------------------------------------------------

Spanish: Si tienes 10 manzanas y comes 3, ¿cuántas te quedan?
Answer: 

Tienes 10 manzanas y comes 3, por lo que te quedan 10 - 3 = <<10-3=7>>7 manzanas.
--------------------------------------------------

French: Si un train roule à 60 km/h pendant 2 heures, quelle distance parcourt-il?
Answer: 

Un train roule à 60 km/h pendant 2 heures, donc il parcourt 60*2 = <<60*2=120>>120 km.
--------------------------------------------------
